In [3]:
%pwd

'c:\\Users\\hp\\Documents\\Projects\\Personal-Information-Chaos-Manager\\backend'

In [2]:
import os
os.chdir("../")

In [4]:
%pip install --upgrade langchain==0.0.353 pypdf sentence-transformers==2.2.2 huggingface-hub==0.20.0 transformers==4.35.2 torch==2.1.2 accelerate==0.25.0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
loader = PyPDFLoader("research/2nd semester marks carks.pdf")
documents = loader.load()

In [8]:
print(documents[0].page_content[:500])

Easwari Engineering College
(  (An Autonomous Institution)  )
Semester End Examination - Provisional Grade Card
Name of Student :KISHORE R
University Seat
Number
:310623205074
Degree :B.Tech - Information Technology
UMIS No. :2303310620521074
Semester : SECOND
Month and
Year
: APRIL_2024
Department : Information
Technology
Sl# Subject Code Subject Name Credits Grade CP
1 231MAB201T ADVANCED CALCULUS AND COMPLEX ANALYSIS 4 A 32
2 231CSC201T PROGRAMMING IN C 3 A 24
3 231GEH201T தமிழம் ெதாழில்ட்ப


In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)
print("Total chunks:", len(chunks))

Total chunks: 5


In [10]:
print(chunks[0].page_content)

Easwari Engineering College
(  (An Autonomous Institution)  )
Semester End Examination - Provisional Grade Card
Name of Student :KISHORE R
University Seat
Number
:310623205074
Degree :B.Tech - Information Technology
UMIS No. :2303310620521074
Semester : SECOND
Month and
Year
: APRIL_2024
Department : Information
Technology
Sl# Subject Code Subject Name Credits Grade CP
1 231MAB201T ADVANCED CALCULUS AND COMPLEX ANALYSIS 4 A 32
2 231CSC201T PROGRAMMING IN C 3 A 24


In [12]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [13]:
import os
HF_TOKEN=os.environ.get('HF_TOKEN')

In [14]:
import os
os.environ["HF_TOKEN"]=HF_TOKEN

In [15]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [16]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [17]:
query = "When is my SGPA?"
docs = vectorstore.similarity_search(query, k=2)

for i, doc in enumerate(docs):
    print(f"\n--- Retrieved Chunk {i+1} ---")
    print(doc.page_content)


--- Retrieved Chunk 1 ---
9 231GES213L ELECTRICAL AND ELECTRONICS ENGINEERING
PRACTICES
2 O 20
Credits earned in the current semester Total 22 185
SGPA (Semester Grade Point Average) 8.55
CGPA (Cumulative Grade Point Average) 8.41
Mandatory Learning Courses (*PP: Passed ,*NP: Not Passed)
Percentage Equivalence of Grade Points (10-Point scale)
Grade Point Class
= 8.5 and
< = 10
FCD
= 7 and
< = 8.49
FC
= 5 and
< = 6.99
SC
= 0 and
< = 4.99
F
Grade O A+ A B+ B C U W U
Grade
Points
10 9 8 7 6 5 0 0 0

--- Retrieved Chunk 2 ---
Easwari Engineering College
(  (An Autonomous Institution)  )
Semester End Examination - Provisional Grade Card
Name of Student :KISHORE R
University Seat
Number
:310623205074
Degree :B.Tech - Information Technology
UMIS No. :2303310620521074
Semester : SECOND
Month and
Year
: APRIL_2024
Department : Information
Technology
Sl# Subject Code Subject Name Credits Grade CP
1 231MAB201T ADVANCED CALCULUS AND COMPLEX ANALYSIS 4 A 32
2 231CSC201T PROGRAMMING IN C 3 A 24


In [18]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Loading model: {model_name}")
print("This may take a minute...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Use float32 for CPU compatibility (float16 not supported for CPU operations)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="cpu",
    low_cpu_mem_usage=True  # Optimize memory usage
)

print("Model loaded successfully!")

Loading model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
This may take a minute...


: 

In [12]:
hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0,
    do_sample=False,
    repetition_penalty=1.1
)


In [13]:
llm = HuggingFacePipeline(pipeline=hf_pipeline)

In [14]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 2}),
    return_source_documents=True
)

In [15]:
question = "When is my SGPA?"

result = qa_chain(question)

print("ANSWER:\n", result["result"])

c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\generation\configuration_utils.py:381: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


RuntimeError: "addmm_impl_cpu_" not implemented for 'Half'

In [ ]:
print("\nSOURCE DOCUMENTS:\n")
for doc in result["source_documents"]:
    print(doc.page_content)
    print("----")

In [ ]:
qa_chain("What is my bank balance?")